# **ASSIGNMENT NO. 10: Final Project – Choosing and Applying a Data Structure**

# **Project – Undo and Redo Text Editor Using Stacks**

# 1.Design Note

The project is a simple text editor that supports inserting text, deleting text, undo and redo operations. The main operations are insert, delete, undo and redo. Undo and redo are performed frequently, so stacks are suitable because the most recent operation must be handled first.

I first use two stacks called undo_stack and redo_stack. In the first version, I store the complete text after every change. This makes undo and redo simple, but it uses more memory because the whole text is stored repeatedly.

In the improved version, I store only the changes made to the text instead of storing the complete text. This can reduce the amount of data stored for large text. The trade-off is that the code becomes slightly more complicated because the changes must be applied and reversed correctly.**

I will test both versions using more than 10,000 operations and measure their execution time. I will compare the results and identify the main bottleneck from the measurements.

# 2. Version 1 — Full Snapshots

In [1]:
# Undo/Redo Text Editor - Version 1
# Stores the complete text in the stacks

class TextEditor:
    def __init__(self):
        self.text = ""
        self.undo_stack = []
        self.redo_stack = []

    # Add text
    def insert(self, text):
        self.undo_stack.append(self.text)
        self.text += text
        self.redo_stack = []

    # Delete last n characters
    def delete(self, n):
        if n <= len(self.text):
            self.undo_stack.append(self.text)
            self.text = self.text[:-n]
            self.redo_stack = []

    # Undo last change
    def undo(self):
        if self.undo_stack:
            self.redo_stack.append(self.text)
            self.text = self.undo_stack.pop()

    # Redo last undone change
    def redo(self):
        if self.redo_stack:
            self.undo_stack.append(self.text)
            self.text = self.redo_stack.pop()

    # Show current text
    def show(self):
        print("Text:", self.text)


# Test
e = TextEditor()

e.insert("Hello")
e.insert(" World")

print("After insert:")
e.show()

e.undo()
print("After undo:")
e.show()

e.redo()
print("After redo:")
e.show()

e.delete(6)
print("After delete:")
e.show()

e.undo()
print("After undo:")
e.show()

After insert:
Text: Hello World
After undo:
Text: Hello
After redo:
Text: Hello World
After delete:
Text: Hello
After undo:
Text: Hello World


# 3. Version 1 Performance Test

In [2]:
import time

e = TextEditor()

start = time.time()

for i in range(10000):
    e.insert("abc")

for i in range(5000):
    e.undo()

for i in range(5000):
    e.redo()

time1 = time.time() - start

print("Version 1 time:", round(time1, 6), "seconds")

Version 1 time: 0.290704 seconds


# 4. Version 2 — Store Changes

In [3]:
# Undo/Redo Text Editor - Version 2
# Stores only the changes

class TextEditor2:
    def __init__(self):
        self.text = ""
        self.undo_stack = []
        self.redo_stack = []

    # Insert text
    def insert(self, text):
        self.text += text
        self.undo_stack.append(("insert", text))
        self.redo_stack = []

    # Delete last n characters
    def delete(self, n):
        if n <= len(self.text):
            deleted = self.text[-n:]
            self.text = self.text[:-n]
            self.undo_stack.append(("delete", deleted))
            self.redo_stack = []

    # Undo
    def undo(self):
        if not self.undo_stack:
            return

        action, value = self.undo_stack.pop()

        if action == "insert":
            self.text = self.text[:-len(value)]
        else:
            self.text += value

        self.redo_stack.append((action, value))

    # Redo
    def redo(self):
        if not self.redo_stack:
            return

        action, value = self.redo_stack.pop()

        if action == "insert":
            self.text += value
        else:
            self.text = self.text[:-len(value)]

        self.undo_stack.append((action, value))

    # Display text
    def show(self):
        print("Text:", self.text)


# Test
e = TextEditor2()

e.insert("Hello")
e.insert(" World")

print("After insert:")
e.show()

e.undo()
print("After undo:")
e.show()

e.redo()
print("After redo:")
e.show()

e.delete(6)
print("After delete:")
e.show()

e.undo()
print("After undo:")
e.show()

After insert:
Text: Hello World
After undo:
Text: Hello
After redo:
Text: Hello World
After delete:
Text: Hello
After undo:
Text: Hello World


# 5. Version 2 Performance Test

In [4]:
import time

e = TextEditor2()

start = time.time()

for i in range(10000):
    e.insert("abc")

for i in range(5000):
    e.undo()

for i in range(5000):
    e.redo()

time2 = time.time() - start

print("Version 2 time:", round(time2, 6), "seconds")

Version 2 time: 0.103173 seconds


# 6. Comparison Table

## 

| Version | Method | Operations | Time |
|---|---|---:|---:|
| Version 1 | Full snapshots | 20,000 | 0.290704 seconds |
| Version 2 | Store changes | 20,000 | 0.103173 seconds |

### Observation

Version 2 was faster than Version 1 in this test. Version 1 took
0.290704 seconds, while Version 2 took 0.103173 seconds. This shows
that storing only the changes reduced the execution time in this test.

## Complexity Comparison

| Operation | Version 1 | Version 2 |
|---|---|---|
| Insert | O(n) | O(n) |
| Delete | O(n) | O(n) |
| Undo | O(n) | O(n) |
| Redo | O(n) | O(n) |

### Explanation

Version 1 stores the complete text after every change, so undo and redo
may require copying the text and take O(n) time.

Version 2 stores only the changes, but changing the string can still take
O(n) time because Python strings are immutable.

# 7. Improvement That Did Not Help




In [7]:
# Test whether doing more undo/redo operations improves the speed

import time

e = TextEditor2()

# Create some text
for i in range(10000):
    e.insert("abc")

# Measure many undo and redo operations
start = time.time()

for i in range(10000):
    e.undo()

for i in range(10000):
    e.redo()

extra_time = time.time() - start

print("Time for extra undo/redo operations:", round(extra_time, 6), "seconds")


Time for extra undo/redo operations: 0.077222 seconds


## Improvement That Did Not Help

I tried increasing the number of undo and redo operations to see whether
it would improve the performance. It did not improve the program because
more operations simply required more work from the stacks. Therefore,
increasing the history size was not a useful performance improvement.

The better improvement was storing only the changes instead of storing
the complete text after every operation.

# 8.Conclusion
The project shows how stacks are useful for implementing undo and redo. The first version stores complete text snapshots, which makes the implementation simple but can use more memory. The second version stores only the changes, which reduces repeated storage. The experiment also shows that the choice of data structure and the way data is stored can affect the performance of a program.

# 9. Short Report

The aim of this project was to create a simple text editor with insert,
delete, undo and redo operations. Since undo and redo always work with the
most recent operation first, stacks are suitable for this problem. I used
two stacks, one for undo operations and one for redo operations.

In Version 1, I stored the complete text after every change. This made the
program simple because undo could restore the previous text directly.
However, storing the complete text repeatedly can use more memory when the
text becomes large.

In Version 2, I stored only the changes made to the text. For example, an
insert operation stores the inserted text and a delete operation stores the
deleted text. This reduces repeated storage and makes the history more
compact. The code is slightly more complicated because each change has to
be reversed correctly during undo.

I tested both versions using 10,000 insert operations, followed by 5,000
undo operations and 5,000 redo operations. Version 1 took 0.290704 seconds,
while Version 2 took 0.103173 seconds. Therefore, Version 2 was faster in
this test. The result shows that the way information is stored can affect
the performance of a program.

I also tried increasing the number of undo and redo operations as an
improvement experiment. The additional test took 0.077222 seconds, but
this did not improve the original program. Increasing the number of
operations only increased the amount of work performed, so it was not a
useful optimization.

The main bottleneck in Version 1 was repeatedly storing and restoring
complete strings. Version 2 avoided storing the complete text in the
history and stored only the required changes. Both versions still use
stacks, showing how the same data structure can be used with different
storage strategies.

Overall, this project demonstrated that choosing a suitable data structure
and storage method is important for performance. The stack was a suitable
choice because undo and redo follow the Last-In-First-Out principle.